# Stage 1B — DAVIS frame-to-frame temporal windows

This stage indexes each consecutive APS pair as a half-open interval `[t_k, t_(k+1))`, with event, IMU, and ground-truth samples selected on the same common timestamp axis. It preserves the original Tonic stream-local integer-microsecond timestamps and uses the validated raw-bag manifest's source-second stream origins only to calculate explicit offsets. Sample arrays are returned as views/index ranges; nothing is copied into every window.

The common axis is relative to the earliest raw stream start. The manifest's source-second origins are converted to integer-microsecond offsets (the resolution of Tonic's arrays); no interpolation, clock correction, or feature processing is performed.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src" / "frame_synchronization.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.davis_inspection import DavisConfig, load_davis
from src.frame_synchronization import (
    load_timestamp_origins,
    make_frame_windows,
    plot_frame_intervals,
    validate_frame_windows,
    write_synchronization_report,
)

config = DavisConfig()
recording = load_davis(config)
origins_s, manifest_info = load_timestamp_origins(recording, config)
windows = make_frame_windows(recording, origins_s)
validation = validate_frame_windows(windows)
print(f"Frame intervals: {len(windows):,}")
print(f"Events in APS span: {validation['events_within_aps_span']:,}")
print(f"Empty event windows: {validation['empty_event_windows']:,}")
print(f"IMU samples in APS span: {validation['imu_samples_within_aps_span']:,}")
print(f"Ground-truth samples in APS span: "
      f"{validation['groundtruth_samples_within_aps_span']:,}")
print(f"Outside APS span: before={validation['events_before_first_aps_frame']:,}, "
      f"after={validation['events_at_or_after_last_aps_frame']:,}")
print(f"Validation passed: {validation['passed']}")
print("Checks:", validation["checks"])

In [ ]:
window = windows.get_window(0)
print("Interval index:", window.index)
print("Common interval [start, end) us:", window.interval_us)
print("Original APS timestamps us:", window.frame_k_timestamp_us,
      window.frame_k_plus_1_timestamp_us)
print("Original event index range:", window.event_index_range)
print("Events in interval:", len(window.events_k))
print("IMU samples:", 0 if window.imu_k is None else len(window.imu_k))
print("Ground-truth samples:", 0 if window.groundtruth_k is None else len(window.groundtruth_k))
print("Event Tonic timestamps (unchanged):", window.events_k["t"][:5])
print("Event common timestamps us:",
      window.events_k["t"][:5] + window.event_timestamp_offset_us)

In [ ]:
output_dir = project_root / "results" / "dataset_synchronization"
plot_path = plot_frame_intervals(
    windows, output_dir / f"{config.recording}_frame_intervals.png", interval_count=5
)
report_path = write_synchronization_report(
    windows,
    validation,
    manifest_info,
    output_dir / f"{config.recording}_synchronization_report.md",
)
print("Plot:", plot_path)
print("Report:", report_path)

## Validation and limits

`validate_frame_windows` verifies half-open event membership, adjacent event-index continuity, no overlap or loss inside the first-to-last APS span, and empty-window support. Events before the first APS timestamp and events at/after the last APS timestamp are reported separately because they do not belong to any consecutive frame interval. IMU and ground-truth timestamps exposed by a window remain in their original stream-local microseconds; `common_timestamps_us` is an explicit mapped view for interval interpretation.

This stage only applies constant per-stream start offsets measured from the raw bag manifest. It does not perform feature detection, event tracking, interpolation, resampling, clock-drift correction, or calibration.